# 02 - Preprocessing: membangun data siap latih

Menjalankan keputusan yang dikunci di `01_eda.ipynb`:

    muat -> buang missing -> resolusi konflik label -> dedup NFKC-exact
    -> stratified split 70/15/15 -> clean_text (SETELAH split)
    -> guard anti-kebocoran -> simpan

Keluaran: `data/processed/{train,val,test}.csv`, `data/processed/metadata.json`,
dan `data/interim/data_clean.csv`.

Urutan operasi menentukan isi split, sehingga mengubahnya membatalkan seluruh
angka eksperimen yang sudah dihasilkan.

In [1]:
import pandas as pd

from src.config import LABEL_COLUMN, RAW_TEXT_COLUMN, settings
from src.services.preprocessing import DatasetBuilder

raw = pd.read_csv(settings.raw_csv, index_col=0)[[RAW_TEXT_COLUMN, LABEL_COLUMN]]
print(f"baris mentah: {len(raw):,}")

builder = DatasetBuilder(seed=settings.random_seed)

baris mentah: 14,237


## 1. Jalankan pipeline

In [2]:
splits = builder.build(raw)

print(builder.counts)
print(builder.label_conflict)
for name, frame in splits.items():
    positif = int((frame[LABEL_COLUMN] == 1).sum())
    print(f"{name:5s}: {len(frame):5,} baris | kelas judi {positif:4,} "
          f"({positif / len(frame) * 100:.2f}%)")

2026-09-02 14:26:09,669 | INFO     | src.services.preprocessing | Dedup NFKC-exact: 14227 -> 9412 baris
2026-09-02 14:26:09,701 | INFO     | src.services.preprocessing | Stratified split (0.7, 0.15, 0.15): train=6588 val=1412 test=1412
2026-09-02 14:26:09,984 | INFO     | src.services.preprocessing | Duplikat text_clean lintas-split dibuang dari val/test: 17 (val 1412 -> 1402, test 1412 -> 1405)
{'raw': 14237, 'after_missing': 14227, 'after_dedup': 9412, 'leakage_removed': 17, 'final_total': 9395}
{'groups': 3, 'rows': 32, 'policy': 'assign_1'}
train: 6,588 baris | kelas judi 1,197 (18.17%)
val  : 1,402 baris | kelas judi  257 (18.33%)
test : 1,405 baris | kelas judi  256 (18.22%)


`leakage_removed` adalah baris val/test yang `text_clean`-nya identik dengan
baris train. Dedup NFKC di awal bekerja pada teks asli, sehingga dua komentar
yang hanya berbeda pada URL atau nominal masih lolos sebagai baris terpisah;
setelah placeholder diterapkan keduanya menjadi identik dan menjadi kebocoran
nyata. Prioritas pembuangan train > val > test, jadi himpunan latih tidak
pernah berkurang.

## 2. Contoh transformasi teks

In [3]:
contoh = splits["train"]
mask = contoh["text_clean"].str.contains(r"\[URL\]|\[MENTION\]|\[NUM\]", regex=True)
for _, row in contoh[mask].head(6).iterrows():
    print(f"  L{row[LABEL_COLUMN]} | RAW  : {str(row[RAW_TEXT_COLUMN])[:95]}")
    print(f"       | CLEAN: {row['text_clean'][:95]}\n")

  L0 | RAW  : Top Up Terbaik Se indonesia Hanya Di https://ourastore.com
       | CLEAN: Top Up Terbaik Se indonesia Hanya Di [URL]

  L1 | RAW  : 11;40 Ini sih harus viral!🤎𝐏𝐑𝐎𝐁𝐄𝐓 𝟖𝟓𝟓🤎
       | CLEAN: [NUM];[NUM] Ini sih harus viral!🤎PROBET [NUM]🤎

  L1 | RAW  : 23;16 Subhanallah,🤎𝐏𝐑𝐎𝐁𝐄𝐓 𝟖𝟓𝟓🤎bikin pengalaman main jadi beda! Ceritanya penuh makna, visualnya
       | CLEAN: [NUM];[NUM] Subhanallah,🤎PROBET [NUM]🤎bikin pengalaman main jadi beda! Ceritanya penuh makna, v

  L0 | RAW  : ​@PulcherEtFortisahh. Mm ama junglernya emang ampas
       | CLEAN: [MENTION]. Mm ama junglernya emang ampas

  L0 | RAW  : ​@ahmadravi6730lah emang nya saya menjawab dengan nada ngajak ribut dan tawuran gitu wkwk
       | CLEAN: [MENTION] emang nya saya menjawab dengan nada ngajak ribut dan tawuran gitu wkwk

  L0 | RAW  : ​​​@apaGAterimaloberakti bkent Epic dong aduh bocil bocil tido esok sekolah dek🤣
       | CLEAN: [MENTION] bkent Epic dong aduh bocil bocil tido esok sekolah dek🤣



## 3. Verifikasi anti-kebocoran

In [4]:
teks = {name: set(frame["text_clean"]) for name, frame in splits.items()}
kunci = {name: set(frame["nfkc_key"]) for name, frame in splits.items()}

for kiri, kanan in (("train", "val"), ("train", "test"), ("val", "test")):
    print(f"{kiri}-{kanan}: nfkc_key {len(kunci[kiri] & kunci[kanan])} | "
          f"text_clean {len(teks[kiri] & teks[kanan])}")

train-val: nfkc_key 0 | text_clean 0
train-test: nfkc_key 0 | text_clean 0
val-test: nfkc_key 0 | text_clean 0


Keenam angka harus nol. Kebocoran train-test membuat retrieval RM-c menemukan
sampel uji di dalam indeksnya sendiri, yang akan melebih-lebihkan hasilnya.

## 4. Class weight

In [5]:
weights = builder.compute_class_weights(splits["train"][LABEL_COLUMN])
print(weights)

{0: 0.6110183639398998, 1: 2.7518796992481205}


Dihitung dari split train saja. Menghitungnya dari seluruh data akan
membocorkan distribusi val/test ke dalam loss.

## 5. Gate reproduktibilitas

In [6]:
import hashlib
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    sementara = Path(tmp)
    builder.write(splits, output_dir=sementara,
                  interim_path=sementara / "data_clean.csv", source=settings.raw_csv)
    baru = {
        name: hashlib.sha256((sementara / f"{name}.csv").read_bytes()).hexdigest()
        for name in ("train", "val", "test")
    }

lama = {}
for name in ("train", "val", "test"):
    path = settings.split_path(name)
    lama[name] = hashlib.sha256(path.read_bytes()).hexdigest() if path.exists() else None

identik = True
for name in ("train", "val", "test"):
    if lama[name] is None:
        print(f"{name:5s}: belum ada split lama, akan dibuat baru")
    else:
        cocok = baru[name] == lama[name]
        identik &= cocok
        print(f"{name:5s}: {'IDENTIK' if cocok else 'BERBEDA'}  "
              f"{baru[name][:16]} vs {lama[name][:16]}")

if any(lama.values()) and not identik:
    raise RuntimeError(
        "Split baru berbeda dengan split yang sudah dipakai eksperimen. "
        "Jangan timpa: periksa dulu apa yang berubah, karena seluruh angka "
        "hasil mengasumsikan pembagian baris yang lama."
    )

2026-09-02 14:26:10,454 | INFO     | src.services.preprocessing | Split dan metadata ditulis ke C:\Users\Arya\AppData\Local\Temp\tmpaff7mmr4
train: IDENTIK  c419bb079d791498 vs c419bb079d791498
val  : IDENTIK  3fb36cd07f326cd5 vs 3fb36cd07f326cd5
test : IDENTIK  41c232d3f6625660 vs 41c232d3f6625660


Sel di atas menghitung split ke folder sementara dan membandingkan SHA-256-nya
dengan berkas yang sudah ada. Menimpa split secara diam-diam akan membuat
seluruh riwayat run menjadi yatim: angka lama merujuk pembagian baris yang tidak
lagi bisa direproduksi.

## 6. Simpan

In [7]:
written = builder.write(splits, source=settings.raw_csv)
for name, path in written.items():
    print(f"{name:9s}: {path}  ({path.stat().st_size / 1024:.1f} KB)")

2026-09-02 14:26:10,864 | INFO     | src.services.preprocessing | Split dan metadata ditulis ke C:\Penelitian\IndoBERT-with-RAC\data\processed
train    : C:\Penelitian\IndoBERT-with-RAC\data\processed\train.csv  (724.8 KB)
val      : C:\Penelitian\IndoBERT-with-RAC\data\processed\val.csv  (155.5 KB)
test     : C:\Penelitian\IndoBERT-with-RAC\data\processed\test.csv  (155.1 KB)
combined : C:\Penelitian\IndoBERT-with-RAC\data\interim\data_clean.csv  (1086.3 KB)
metadata : C:\Penelitian\IndoBERT-with-RAC\data\processed\metadata.json  (1.3 KB)


In [8]:
import json

metadata = json.loads(settings.metadata_path.read_text(encoding="utf-8"))
print(json.dumps({k: metadata[k] for k in ("counts", "label_conflict", "class_weights")},
                 indent=2, ensure_ascii=False))

{
  "counts": {
    "raw": 14237,
    "after_missing": 14227,
    "after_dedup": 9412,
    "leakage_removed": 17,
    "final_total": 9395
  },
  "label_conflict": {
    "groups": 3,
    "rows": 32,
    "policy": "assign_1"
  },
  "class_weights": {
    "0": 0.6110183639398998,
    "1": 2.7518796992481205
  }
}


## Ringkasan

Input model adalah kolom `text_clean`, bukan `textOriginal`. Karena preprocessing
menambahkan tiga special token, setiap model wajib memanggil
`resize_token_embeddings(len(tokenizer))`; hal itu sudah ditangani oleh factory
di `src/models/heads.py`.

Lanjut ke `03a_rma_finetune.ipynb`.